# Llama-3.2 Korean Bllossom 3B — evaluation

## 1. Install dependencies

In [ ]:
!pip install transformers accelerate trl
!pip install --upgrade datasets

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Paths & config

In [ ]:
import os, sys

DRIVE_ROOT = '/content/drive/MyDrive'
BASE_DIR   = f'{DRIVE_ROOT}/capstone_design'
SRC_DIR    = f'{BASE_DIR}/llm_src'
HF_REPO    = 'minsu0567/Capstone-Design-Llama3.2-Command'
TEST_JSONL = f'{BASE_DIR}/data/test.jsonl'
PAD_TOKEN  = '<|reserved_special_token_247|>'

MAX_NEW_TOKENS = 128

assert os.path.isfile(f'{SRC_DIR}/llama3_2_sft.py'), f'Missing: {SRC_DIR}/llama3_2_sft.py'
assert os.path.isfile(TEST_JSONL), f'Missing: {TEST_JSONL}'

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print('Paths OK.')

## 4. Load merged model from Hugging Face

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

model = AutoModelForCausalLM.from_pretrained(HF_REPO, torch_dtype=DTYPE, device_map='auto')
model.eval()

tokenizer = AutoTokenizer.from_pretrained(HF_REPO)
if tokenizer.pad_token_id is None:
    tokenizer.add_special_tokens({'pad_token': PAD_TOKEN})
model.config.pad_token_id = tokenizer.pad_token_id

print('Loaded', HF_REPO, 'on', model.device, DTYPE)

## 5. Holdout test set

In [ ]:
from datasets import load_dataset

eval_dataset = load_dataset('json', data_files={'test': TEST_JSONL}, split='test')

print(f'테스트 데이터셋 크기: {len(eval_dataset)}')
print(eval_dataset[0])

## 6. Evaluate

In [ ]:
from llama3_2_sft import evaluate_model, print_results

INSTRUCTION = '사용자 입력을 보고, 제어 명령을 생성해주세요. 오직 JSON 객체만 생성하세요.'

results, accuracy = evaluate_model(
    eval_dataset, INSTRUCTION, model, tokenizer, max_new_tokens=MAX_NEW_TOKENS)

print_results(results, limit=20)
print(f'
정확도: {accuracy * 100:.2f}%')